In [33]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [34]:
class FDAEPreprocessor:
    """
    Preprocessing pipeline for the Fraud Detection Autoencoder (FDAE).

    The FDAE is an unsupervised anomaly detector — it learns what *normal*
    looks like, then flags anything that doesn't reconstruct well as fraud.
    This means:
      - Training: normal-only transactions
      - Validation: normal-only (to tune thresholds without leaking fraud signal)
      - Test: full dataset (fraud + normal) for evaluation
    """

    def __init__(self):
        self.scaler = RobustScaler()   # robust to the extreme outliers common in fraud data
        self.feature_columns = None
        self.target_column = 'Class'
        self.n_features = None

    # ------------------------------------------------------------------
    # Data loading
    # ------------------------------------------------------------------

    def load_data(self, filepath):
        df = pd.read_csv(filepath)
        print(f'Loaded {len(df):,} transactions')
        print(f'  Normal (0): {(df[self.target_column] == 0).sum():,}')
        print(f'  Fraud  (1): {(df[self.target_column] == 1).sum():,}')
        print(f'  Fraud rate: {df[self.target_column].mean():.4%}')
        return df

    # ------------------------------------------------------------------
    # Feature engineering  

    def engineer_features(self, df):
        """
        Adds time, amount, and interaction features.
        The interaction terms help the autoencoder learn joint patterns
        that fraud exploits — matching the wider encoder design.
        """
        df = df.copy()

        # Time-based
        df['Hour'] = (df['Time'] % (24 * 3600)) // 3600
        df['Day']  = df['Time'] // (24 * 3600)

        # Amount-based 
        df['Amount_log']  = np.log1p(df['Amount'])
        df['Amount_sqrt'] = np.sqrt(df['Amount'])

        # Pairwise interactions among the most discriminative V-features
        top_features = ['V14', 'V4', 'V11', 'V12', 'V10']
        for i, f1 in enumerate(top_features):
            for f2 in top_features[i + 1:]:
                df[f'{f1}_{f2}_interaction'] = df[f1] * df[f2]

        return df



    def prepare_splits(self, df, test_size=0.2, val_size=0.2, random_state=42):
        """
        Produces three splits:
          X_train / y_train  — normal only  (FDAE training)
          X_val   / y_val    — normal only  (threshold tuning, loss monitoring)
          X_test  / y_test   — full dataset (evaluation against fraud labels)

        The test set is split first from the full dataset (stratified) so both
        classes are represented proportionally.  Train/val are then filtered
        to normal-only.
        """
        X = df.drop(columns=[self.target_column])
        y = df[self.target_column]
        self.feature_columns = X.columns.tolist()
        self.n_features = len(self.feature_columns)

        # 1. Stratified test split from the full dataset
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )

        # 2. Filter temp to normal-only before the train/val split
        normal_mask = y_temp == 0
        X_normal = X_temp[normal_mask]
        y_normal = y_temp[normal_mask]

        # 3. Train / val split on normal-only data
        adjusted_val = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_normal, y_normal,
            test_size=adjusted_val,
            random_state=random_state
            
        )



        print('\nSplit summary:')
        print(f'  Train : {len(X_train):,} normal transactions')
        print(f'  Val   : {len(X_val):,} normal transactions')
        print(f'  Test  : {len(X_test):,} total  '
              f'({(y_test == 1).sum()} fraud, {(y_test == 0).sum()} normal)')

        return X_train, X_val, X_test, y_train, y_val, y_test


    def prepare_stage2_splits(self,df,test_size=0.2,val_size=0.2,random_state=42):
        """
        Supervised splits for the stage-2 classifier.
    
        Unlike the FDAE splits, these retain BOTH fraud and normal
        transactions because the classifier is supervised.
    
        Returns:
            X_train, X_val, X_test
            y_train, y_val, y_test
        """
    
        X = df.drop(columns=[self.target_column])
        y = df[self.target_column]
    
        # First split off test set
        X_temp, X_test, y_temp, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            stratify=y,
            random_state=random_state
        )
    
        # Then split remaining into train/val
        adjusted_val = val_size / (1 - test_size)
    
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp,
            y_temp,
            test_size=adjusted_val,
            stratify=y_temp,
            random_state=random_state
        )
    
        print('\nStage-2 supervised split summary:')
        print(f'  Train : {len(X_train):,} '
              f'({(y_train == 1).sum()} fraud)')
        print(f'  Val   : {len(X_val):,} '
              f'({(y_val == 1).sum()} fraud)')
        print(f'  Test  : {len(X_test):,} '
              f'({(y_test == 1).sum()} fraud)')
    
        return (X_train,X_val,X_test,y_train,y_val,y_test)

    def scale_stage2_features(self,X_train,X_val,X_test):
        """
        Scale stage-2 supervised splits using the SAME scaler
        fitted on FDAE normal training data.
        """
    
        X_train_s = pd.DataFrame(
            self.scaler.transform(X_train),
            columns=self.feature_columns,
            index=X_train.index
        )
    
        X_val_s = pd.DataFrame(
            self.scaler.transform(X_val),
            columns=self.feature_columns,
            index=X_val.index
        )
    
        X_test_s = pd.DataFrame(
            self.scaler.transform(X_test),
            columns=self.feature_columns,
            index=X_test.index
        )
    
        return X_train_s, X_val_s, X_test_s

    # ------------------------------------------------------------------
    # Scaling
    # ------------------------------------------------------------------

    def scale_features(self, X_train, X_val, X_test):
        """
        Fit scaler on normal training data only — consistent with the
        unsupervised setup.  RobustScaler is preferred over StandardScaler
        because fraud amounts create extreme outliers.
        """
        X_train_s = pd.DataFrame(
            self.scaler.fit_transform(X_train),
            columns=self.feature_columns, index=X_train.index
        )
        X_val_s = pd.DataFrame(
            self.scaler.transform(X_val),
            columns=self.feature_columns, index=X_val.index
        )
        X_test_s = pd.DataFrame(
            self.scaler.transform(X_test),
            columns=self.feature_columns, index=X_test.index
        )
        print(f'\nScaling complete. Feature count: {self.n_features}')
        return X_train_s, X_val_s, X_test_s

    # ------------------------------------------------------------------
    # Serialisation
    # ------------------------------------------------------------------

    def save(self, filepath='../processed/preprocessor_fdae.joblib'):
        joblib.dump({
            'scaler': self.scaler,
            'feature_columns': self.feature_columns,
            'n_features': self.n_features
        }, filepath)
        print(f'Preprocessor saved  {filepath}')

    def load(self, filepath='preprocessor_fdae.joblib'):
        data = joblib.load(filepath)
        self.scaler = data['scaler']
        self.feature_columns = data['feature_columns']
        self.n_features = data['n_features']

In [35]:
def add_gaussian_noise(x: torch.Tensor, noise_factor: float = 0.1) -> torch.Tensor:
    """
    Corrupt input tensor with additive Gaussian noise.

    Args:
        x:            Input tensor  (batch_size, n_features)
        noise_factor: Std of the noise relative to the signal. 
                      0.1 is a good starting point; increase if the model
                      overfits, decrease if training loss is too high.

    Returns:
        Noisy tensor of same shape as x.
    """
    noise = torch.randn_like(x) * noise_factor
    return x + noise


# Quick sanity check
dummy = torch.randn(4, 30)
noisy = add_gaussian_noise(dummy, noise_factor=0.1)
diff = (noisy - dummy).abs().mean().item()
print(f'Noise sanity check — mean absolute deviation: {diff:.4f} (expect ~0.08)')

Noise sanity check — mean absolute deviation: 0.0770 (expect ~0.08)


In [36]:
 def make_dataloaders(
    X_train, X_val, X_test, y_test,
    batch_size: int = 256,
    device: torch.device = torch.device('cpu')
):
    """
    Build DataLoaders for the FDAE training pipeline.

    Returns:
        train_loader  — shuffled, normal only, features only
        val_loader    — unshuffled, normal only, features only
        test_loader   — unshuffled, full dataset, (features, labels)
    """
    def to_tensor(df):
        return torch.tensor(df.values, dtype=torch.float32)

    X_train_t = to_tensor(X_train)
    X_val_t   = to_tensor(X_val)
    X_test_t  = to_tensor(X_test)
    y_test_t  = torch.tensor(y_test.values, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_train_t),
        batch_size=batch_size,
        shuffle=True,         
        drop_last=True         
    )

    val_loader = DataLoader(
        TensorDataset(X_val_t),
        batch_size=batch_size,
        shuffle=False
    )

    # Test loader includes labels for evaluation
    test_loader = DataLoader(
        TensorDataset(X_test_t, y_test_t),
        batch_size=batch_size,
        shuffle=False
    )

    print('\nDataLoader summary:')
    print(f'  train_loader : {len(train_loader):,} batches * {batch_size}')
    print(f'  val_loader   : {len(val_loader):,} batches * {batch_size}')
    print(f'  test_loader  : {len(test_loader):,} batches * {batch_size}')

    return train_loader, val_loader, test_loader

 def extract_fdae_features(model, dataloader):
    """
    Extract rich FDAE representations for stage-2 classification.

    Returns:
        X_features
        y_labels
    """

    model.eval()

    all_features = []
    all_labels = []

    with torch.no_grad():

        for batch in dataloader:

            x, y = batch

            x_hat, z = model(x)

            recon_error = ((x_hat - x) ** 2).mean(dim=1)
            latent_dist = (z ** 2).mean(dim=1)

            combined = torch.cat([
                recon_error.unsqueeze(1),
                latent_dist.unsqueeze(1),
                z
            ], dim=1)

            all_features.append(combined.numpy())
            all_labels.append(y.numpy())

    X = np.concatenate(all_features)
    y = np.concatenate(all_labels)

    return X, y


In [37]:
def run_preprocessing(filepath: str, batch_size: int = 256):
    """
    End-to-end preprocessing for FDAE.

    Returns:
        train_loader, val_loader, test_loader  — ready for the model training loop
        preprocessor                           — fitted scaler + metadata
        n_features                             — input dimension for model init
    """
    prep = FDAEPreprocessor()

    # Load
    df = prep.load_data('../data/raw/creditcard.csv')

    # Feature engineering
    df = prep.engineer_features(df)

    # Splits  (normal-only train/val, full test)
    X_train, X_val, X_test, y_train, y_val, y_test = prep.prepare_splits(df)

    # Scale
    X_train_s, X_val_s, X_test_s = prep.scale_features(X_train, X_val, X_test)

    X_s2_train, X_s2_val, X_s2_test, y_s2_train, y_s2_val, y_s2_test = \
        prep.prepare_stage2_splits(df)
    X_s2_train_s, X_s2_val_s, X_s2_test_s = \
        prep.scale_stage2_features(X_s2_train, X_s2_val, X_s2_test)

    # DataLoaders
    train_loader, val_loader, test_loader = make_dataloaders(
        X_train_s, X_val_s, X_test_s, y_test,
    batch_size=batch_size,
    device=DEVICE
    )

    # Save preprocessor
    prep.save('../data/processed/preprocessor_fdae.joblib')
    
    joblib.dump({
        'X_s2_train_s': X_s2_train_s, 'y_s2_train': y_s2_train,
        'X_s2_val_s':   X_s2_val_s,   'y_s2_val':   y_s2_val,
        'X_s2_test_s':  X_s2_test_s,  'y_s2_test':  y_s2_test
    }, '../data/processed/stage2_splits.joblib')
    print('Stage-2 splits saved.')

    print(f'\nReady. Input dimension for model: {prep.n_features}')
    return train_loader, val_loader, test_loader, prep


# Run
train_loader, val_loader, test_loader, preprocessor = run_preprocessing(
    filepath='../data/raw/creditcard.csv',
    batch_size=256
)

N_FEATURES = preprocessor.n_features
print(f'\nN_FEATURES = {N_FEATURES}  (pass this to the FDAE model constructor)')

Loaded 284,807 transactions
  Normal (0): 284,315
  Fraud  (1): 492
  Fraud rate: 0.1727%

Split summary:
  Train : 170,588 normal transactions
  Val   : 56,863 normal transactions
  Test  : 56,962 total  (98 fraud, 56864 normal)

Scaling complete. Feature count: 44

Stage-2 supervised split summary:
  Train : 170,883 (295 fraud)
  Val   : 56,962 (99 fraud)
  Test  : 56,962 (98 fraud)

DataLoader summary:
  train_loader : 666 batches * 256
  val_loader   : 223 batches * 256
  test_loader  : 223 batches * 256
Preprocessor saved  ../data/processed/preprocessor_fdae.joblib
Stage-2 splits saved.

Ready. Input dimension for model: 44

N_FEATURES = 44  (pass this to the FDAE model constructor)


In [38]:
# -- Train loader: features only --
batch = next(iter(train_loader))
X_batch = batch[0]
print(f'Train batch shape   : {X_batch.shape}')   # (256, N_FEATURES)

# Noise corruption for denoising loss
X_noisy = add_gaussian_noise(X_batch, noise_factor=0.1)
print(f'Noisy batch shape   : {X_noisy.shape}')   # same

# -- Test loader: features + labels --
test_batch = next(iter(test_loader))
X_t, y_t = test_batch
print(f'Test batch X shape  : {X_t.shape}')
print(f'Test batch y shape  : {y_t.shape}')
print(f'Fraud in test batch : {y_t.sum().int().item()}')

print('\nAll checks passed. Pipeline ready for FDAE model building.')

Train batch shape   : torch.Size([256, 44])
Noisy batch shape   : torch.Size([256, 44])
Test batch X shape  : torch.Size([256, 44])
Test batch y shape  : torch.Size([256])
Fraud in test batch : 0

All checks passed. Pipeline ready for FDAE model building.
